# 외부 pod 호출하기
- 성공! 
    - ssh 백그라운드 실행명령 이용함
    - 도커파일 -> 이미지  -> pod 생성시 해당 이미지 이용하여 pod 실행에 따라 자동으로 vllm 서빙을 하도록 설정하는 방향으로 감

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://j93jaxl2w0uyaq-7860.proxy.runpod.net/v1",
    api_key="EMPTY"
)

# 테스트
response = client.chat.completions.create(
    model="google/gemma-3-1b-it",
    messages=[
        {"role": "user", "content": "안녕하세요! 파이썬으로 피보나치 수열을 생성하는 코드를 작성해주세요."}
    ],
    temperature=0.7,
    max_tokens=500
)

print(response.choices[0].message.content)

## 파이썬으로 피보나치 수열 생성 코드

다음은 파이썬으로 피보나치 수열을 생성하는 몇 가지 방법입니다.

**1. 반복문 사용 (가장 기본적인 방법)**

```python
def fibonacci_iterative(n):
  """
  반복문을 사용하여 피보나치 수열을 생성합니다.

  Args:
    n: 생성할 피보나치 수열의 항의 개수.

  Returns:
    피보나치 수열의 리스트.
  """
  if n <= 0:
    return []
  elif n == 1:
    return [0]
  else:
    list_fib = [0, 1]
    while len(list_fib) < n:
      next_fib = list_fib[-1] + list_fib[-2]
      list_fib.append(next_fib)
    return list_fib

# 예시
num_terms = 10
fib_sequence = fibonacci_iterative(num_terms)
print(f"피보나치 수열 (최대 {num_terms} 항): {fib_sequence}")
```

**코드 설명:**

*   `fibonacci_iterative(n)` 함수는 `n` 개의 피보나치 수열 항을 생성합니다.
*   `n`이 0보다 작거나 같으면 빈 리스트를 반환합니다.
*   `n`이 1이면 `[0]`을 반환합니다.
*   그렇지 않으면 `[0, 1]` 리스트를 초기화하고, `while` 루프를 사용하여 `list_fib` 리스트의 길이를 `n`으로 줄여나가며 피보나치 수열의 항을 생성합니다.
*   마지막에 `list_fib`를 반환합니다.

**2. 재귀 함수 사용 (가장 간단하지만 효율적이지 않음)**

```python
def fibonacci_recursive(n):
  """
  재귀 함수를 사용하여 피보나치 수열을 생성합니다.

  Args:
    n: 생성할 피보나치 수열의 항의 개수.

  Returns:
    피보나치

# qlora 붙여서  vllm으로 서빙하기
- 베이스와 로라를 붙인 llm 모두 서빙 가능함

## baseLLM 부르기

In [9]:
from openai import OpenAI

client = OpenAI(
    base_url="https://8m8ldnjbsw99ka-7804.proxy.runpod.net/v1",
    api_key="EMPTY",
)

resp = client.chat.completions.create(
    model="google/gemma-3-4b-it",
    messages=[{"role": "user", "content": """
    {
  "assay": "IHC",
  "target": "Hyaluronan",
  "dose": "0.0375 mpk",
  "time_points": ["0.5h", "1h", "2h", "4h"],
  "groups": ["Vehicle", "L19", "L19-WT", "L19-63", "WT-Fc", "63-Fc"],
  "observations": [
    {
      "time": "1h",
      "group": "L19-WT",
      "HA_positive_pixels": "~5%",
      "significance_vs_vehicle": "*"
    },
    {
      "time": "2h",
      "group": "L19-WT",
      "HA_positive_pixels": "~3%",
      "significance_vs_vehicle": "*"
    }
  ]
}

이 실험 결과 해석해줘
    """}],
    max_tokens=1024,
)
print(resp.choices[0].message.content)


이 실험 결과는 Hyaluronan에 대한 치료 효과를 평가하기 위한 IHC (면역 조직 화학) 분석 결과입니다. 주요 내용은 다음과 같습니다.

**실험 개요:**

* **대상:** Hyaluronan (하이드로이용한산)
* **투여 용량:** 0.0375 mpk (mg/kg) - 동물에게 투여되는 용량
* **시간:** 0.5시간, 1시간, 2시간, 4시간
* **그룹:**
    * **Vehicle:** 약물 투여 전, 플라보이터 (vehicle)만 투여된 그룹 (대조군)
    * **L19:** Hyaluronan 분해 효소 L19를 투여한 그룹
    * **L19-WT:** Hyaluronan 분해 효소 L19의 유전자 변형된 (WT, Wild Type) 버전 투여 그룹
    * **L19-63:** Hyaluronan 분해 효소 L19의 특정 부위 (63)에 결합하는 버전 투여 그룹
    * **WT-Fc:** Hyaluronan 분해 효소 L19의 유전자 변형된 버전의 Fc 단백질 (Fc fragment) 투여 그룹
    * **63-Fc:** Hyaluronan 분해 효소 L19의 특정 부위 (63)에 결합하는 버전의 Fc 단백질 투여 그룹

**주요 결과:**

* **L19-WT 그룹:** 1시간 및 2시간에 Hyaluronan 양성 픽셀 (%)이 각각 약 5% 및 3%로 감소했습니다. 이 감소는 **Vehicle 그룹에 비해 통계적으로 유의미**합니다 (별표 *). 즉, L19-WT 그룹에서 Hyaluronan 분해 효소의 활성화가 증가하여 Hyaluronan이 감소했다는 것을 의미합니다.

**해석 및 의미:**

* **Hyaluronan 분해 효소 활성:** L19-WT 그룹은 Hyaluronan 분해 효소의 활성화가 증가하여 하이드로이용한산의 양이 감소함을 보여줍니다. 이는 L19 유전자의 WT 버전이 Hyaluronan 분해에 효과적임을 시사합니다.
* **특정 부위 결합 Fc 단백질의 효과:** WT-Fc 및 63-F

In [18]:
from openai import OpenAI

client = OpenAI(
    base_url="https://8m8ldnjbsw99ka-7804.proxy.runpod.net/v1",
    api_key="EMPTY",
)

resp = client.chat.completions.create(
    model="enapeace_qlora",
    messages=[{"role": "user", "content": """
    {
  "assay": "IHC",
  "target": "Hyaluronan",
  "dose": "0.0375 mpk",
  "time_points": ["0.5h", "1h", "2h", "4h"],
  "groups": ["Vehicle", "L19", "L19-WT", "L19-63", "WT-Fc", "63-Fc"],
  "observations": [
    {
      "time": "1h",
      "group": "L19-WT",
      "HA_positive_pixels": "~5%",
      "significance_vs_vehicle": "*"
    },
    {
      "time": "2h",
      "group": "L19-WT",
      "HA_positive_pixels": "~3%",
      "significance_vs_vehicle": "*"
    }
  ]
}

이 실험 결과 해석해줘
    """}],
    max_tokens=1024,
)
print(resp.choices[0].message.content)


이 실험 결과는 Hyaluronan(HA) 음성 세포에서 HA 양성 세포의 변화를 나타냅니다. L19-WT 군에서 1시간 및 2시간에 HA 양성 세포의 비율이 감소했습니다. 이 감소는 통계적으로 유의미했으며, Vehicle 군에 비해 유의미한 차이를 보였습니다.

**핵심 결과:**

*   **L19-WT 군:** 1시간과 2시간에 HA 양성 세포 비율이 감소했습니다.
*   **통계적 유의미성:** 1시간과 2시간에 감소는 통계적으로 유의미했습니다(星 표시).

**결론:**

L19-WT 군에서 1시간과 2시간 동안 HA 양성 세포의 비율이 감소한 것으로 나타났습니다. 이는 Hyaluronan 유도 HA 양성 세포 변화에 대한 L19-WT 군의 특성을 보여줍니다.

**참고:**

*   L19-WT 군은 Hyaluronan 유도 HA 양성 세포를 갖는 세포 군입니다.
*   Vehicle 군은 효과를 비교하기 위한 대조군입니다.

이 결과는 Hyaluronan과 관련된 세포 변화를 연구하는 데 유용하며, L19-WT 군의 특성을 이해하는 데 도움이 됩니다.


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

import os

SLLM_BASE_URL = os.getenv("SLLM_BASE_URL")
RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME")

print("모든 변수가 있습니다.")

if not SLLM_BASE_URL:
    raise ValueError("❌ SLLM_BASE_URL not found in environment variables (.env)")
if not RUNPOD_API_KEY:
    raise ValueError("❌ RUNPOD_API_KEY not found in environment variables (.env)")
if not MODEL_NAME:
    raise ValueError("❌ MODEL_NAME not found in environment variables (.env)")

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

SLLM_BASE_URL = os.getenv("SLLM_BASE_URL")
RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY", "EMPTY")  # 보통 EMPTY로도 됨
MODEL_NAME = os.getenv("MODEL_NAME")

client = OpenAI(
    base_url=SLLM_BASE_URL,
    api_key=RUNPOD_API_KEY
)

# 테스트
response = client.chat.completions.create(
    model=MODEL_NAME ,
    messages=[
        {"role": "user", "content": """
        {
  "assay": "IHC",
  "target": "Hyaluronan",
  "dose": "0.0375 mpk",
  "time_points": ["0.5h", "1h", "2h", "4h"],
  "groups": ["Vehicle", "L19", "L19-WT", "L19-63", "WT-Fc", "63-Fc"],
  "observations": [
    {
      "time": "1h",
      "group": "L19-WT",
      "HA_positive_pixels": "~5%",
      "significance_vs_vehicle": "*"
    },
    {
      "time": "2h",
      "group": "L19-WT",
      "HA_positive_pixels": "~3%",
      "significance_vs_vehicle": "*"
    }
  ]
}

이 실험 결과 해석해줘
        """}
    ],
    temperature=0.7,
    max_tokens=1024
)

print(response.choices[0].message.content)


### 잘 나옴옴

NotFoundError: Error code: 404